In [1]:
import os
import subprocess
import pandas as pd

In [2]:
def get_dict_of_seq(fasta_file):
    """ function that converts a fasta file to a dictionnary of sequences
    fasta_file: the input fasta file
    """
    
    file_fasta_dict = {}
    # output dict of imported seqs
    
    with open(fasta_file, 'r') as fasta:    
        for line in fasta:
            # loops through the file

            if line.startswith('>') == True:
                seq_info = line.strip('>').strip('\n').split('\t')[0]
                file_fasta_dict[seq_info] = ''
                # checks if seq header

            else:
                file_fasta_dict[seq_info] += line.strip('\n')
                # If not, append nt to current seq
                
    return file_fasta_dict

In [3]:
###Load file description from MiSeq###

df_file = pd.read_csv('/Users/fredb/OneDrive - Université Laval/LandryLab/Projet/Duplication_dosage_paper/Data/Fitness_Functions/Read_processing/Plan_Screen_Conditions.csv', sep=';')
df_file.head()

,SEQ_SAMPLE,Sample_desc,R1_file,R2_file,N_forward,N_reverse
0,P1_A1_T0,P1_A1_T0,P1_A1_T0_R1.fastq.gz,P1_A1_T0_R2.fastq.gz,aactgcatttttttcacatc,ttaagaaacagccgtaacc
1,P1_A2_T0,P1_A2_T0,P1_A2_T0_R1.fastq.gz,P1_A2_T0_R2.fastq.gz,aactgcatttttttcacatc,ttaagaaacagccgtaacc
2,P1_A3_T0,P1_A3_T0,P1_A3_T0_R1.fastq.gz,P1_A3_T0_R2.fastq.gz,aactgcatttttttcacatc,ttaagaaacagccgtaacc
3,P1_A4_T0,P1_A4_T0,P1_A4_T0_R1.fastq.gz,P1_A4_T0_R2.fastq.gz,aactgcatttttttcacatc,ttaagaaacagccgtaacc
4,P1_A5_T0,P1_A5_T0,P1_A5_T0_R1.fastq.gz,P1_A5_T0_R2.fastq.gz,aactgcatttttttcacatc,ttaagaaacagccgtaacc


In [6]:
df_file['Merged_file'] = ''

for ind in list(df_file.index):
    # Convert file paths to WSL format
    r1_path = "/mnt/c/Users/fredb/OneDrive - Université Laval/LandryLab/Projet/Data/NGS_data/" + df_file.loc[ind]['R1_file']
    r2_path = "/mnt/c/Users/fredb/OneDrive - Université Laval/LandryLab/Projet/Data/NGS_data/" + df_file.loc[ind]['R2_file']
    output_path = "/mnt/c/Users/fredb/OneDrive - Université Laval/LandryLab/Projet/Data/Screen_Conditions/Merged_reads_Conditions/" + df_file.loc[ind]['SEQ_SAMPLE'] + '.fasta'
    
    # Construct the pandaseq command to run in WSL
    panda_seq_call = (
        f'wsl pandaseq -f "{r1_path}" -r "{r2_path}" '
        f'-O 625 -k 4 -B -t 0.5 -T 2 '
        f'-p {df_file.loc[ind]["N_forward"]} '
        f'-q {df_file.loc[ind]["N_reverse"]} '
        f'-w "{output_path}"'
    )
    
    try:
        # Execute the command using subprocess
        subprocess.check_output(panda_seq_call, shell=True, stderr=subprocess.DEVNULL)
        # Update the 'Merged_file' column in the DataFrame
        df_file.at[ind, 'Merged_file'] = f'{df_file.loc[ind]["SEQ_SAMPLE"]}.fasta'
    except subprocess.CalledProcessError as e:
        print(f"Error running pandaseq for {df_file.loc[ind]['SEQ_SAMPLE']}: {e}")

In [14]:
df_file.head()

,SEQ_SAMPLE,Sample_desc,R1_file,R2_file,N_forward,N_reverse,Merged_file,Agg_file
0,P1_A1_T0,P1_A1_T0,P1_A1_T0_R1.fastq.gz,P1_A1_T0_R2.fastq.gz,aactgcatttttttcacatc,ttaagaaacagccgtaacc,P1_A1_T0.fasta,P1_A1_T0_agg.fasta
1,P1_A2_T0,P1_A2_T0,P1_A2_T0_R1.fastq.gz,P1_A2_T0_R2.fastq.gz,aactgcatttttttcacatc,ttaagaaacagccgtaacc,P1_A2_T0.fasta,P1_A2_T0_agg.fasta
2,P1_A3_T0,P1_A3_T0,P1_A3_T0_R1.fastq.gz,P1_A3_T0_R2.fastq.gz,aactgcatttttttcacatc,ttaagaaacagccgtaacc,P1_A3_T0.fasta,P1_A3_T0_agg.fasta
3,P1_A4_T0,P1_A4_T0,P1_A4_T0_R1.fastq.gz,P1_A4_T0_R2.fastq.gz,aactgcatttttttcacatc,ttaagaaacagccgtaacc,P1_A4_T0.fasta,P1_A4_T0_agg.fasta
4,P1_A5_T0,P1_A5_T0,P1_A5_T0_R1.fastq.gz,P1_A5_T0_R2.fastq.gz,aactgcatttttttcacatc,ttaagaaacagccgtaacc,P1_A5_T0.fasta,P1_A5_T0_agg.fasta


In [12]:
###Call to get the number of read per unique sequence###

df_file['Agg_file'] = ''

for ind in list(df_file.index):
    vsearch_aggregate_call = (
        r'C:/Users/fredb/Documents/vsearch-2.30.0-win-x86_64/vsearch-2.30.0-win-x86_64/bin/vsearch '
        f'--derep_fulllength "C:/Users/fredb/OneDrive - Université Laval/LandryLab/Projet/Data/Screen_Conditions/Merged_reads_Conditions/{df_file.loc[ind]["Merged_file"]}" '
        f'--relabel seq --output "C:/Users/fredb/OneDrive - Université Laval/LandryLab/Projet/Data/Screen_Conditions/Aggregated_reads_Conditions/{df_file.loc[ind]["SEQ_SAMPLE"]}_agg.fasta" '
        '--sizeout'
    )

    try:
        # Capture both stdout and stderr for debugging
        result = subprocess.check_output(vsearch_aggregate_call, shell=True, stderr=subprocess.PIPE, text=True)
        df_file.at[ind, 'Agg_file'] = str(df_file.loc[ind]['SEQ_SAMPLE']) + '_agg.fasta'
    except subprocess.CalledProcessError as e:
        print(f"Error running vsearch for index {ind}: {e}")
        print(f"Command: {e.cmd}")
        print(f"Return code: {e.returncode}")
        print(f"Error output: {e.stderr}")

In [13]:
df_file.to_csv("C:/Users/fredb/OneDrive - Université Laval/LandryLab/Projet/Data/Screen_Conditions/outputfile_Conditions.csv", sep='\t')